In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
Data = pd.read_csv('customer_churn_messy_raw.csv')

In [4]:
Data.head()

,customer_id,Age,Tenure_Months,Plan,MonthlySpend,SupportTickets_90D,MonthlyUsage_Hours,PaymentHistory_Score,LoginFrequency_30D,FailedPayments_90D,ContractType,AutoRenew,DiscountPercent,DaysSinceLastContact,Complaints_90D,Region,SignupDate,Churn
0,104914,49.0,999.0,Basic,4912.97,3.0,16.85,84.78,11.0,0.0,Yearly,No,5.12,146.0,0.0,West,2025-12-15 00:00:00,No
1,105685,57.0,13.7,Basic,2307.58,1.0,93.98,95.42,13.0,2.0,Yearly,Yes,18.23,146.0,0.0,Unknown,2022-06-22 00:00:00,No
2,101670,41.0,47.2,Basic,984.95,2.0,40.27,60.04,4.0,0.0,Monthly,NaN,4.20,41.0,0.0,south,2020-04-04 00:00:00,No
3,105647,59.0,148.5,Premium,5447.63,3.0,92.35,95.37,11.0,1.0,Annual,Yes,28.17,36.0,0.0,Unknown,2019-05-26 00:00:00,No
4,102490,40.0,17.2,Premium,1348.48,2.0,38.19,79.40,3.0,1.0,Monthly,Yes,4.33,81.0,1.0,North,2021-11-02 00:00:00,No


In [5]:
Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6625 entries, 0 to 6624
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           6625 non-null   int64  
 1   Age                   6606 non-null   object 
 2   Tenure_Months         6625 non-null   float64
 3   Plan                  6220 non-null   object 
 4   MonthlySpend          6618 non-null   object 
 5   SupportTickets_90D    6625 non-null   float64
 6   MonthlyUsage_Hours    6590 non-null   float64
 7   PaymentHistory_Score  6590 non-null   float64
 8   LoginFrequency_30D    6605 non-null   float64
 9   FailedPayments_90D    6625 non-null   float64
 10  ContractType          6232 non-null   object 
 11  AutoRenew             5792 non-null   object 
 12  DiscountPercent       6625 non-null   float64
 13  DaysSinceLastContact  6589 non-null   float64
 14  Complaints_90D        6625 non-null   float64
 15  Region               

In [6]:
for col in ['Age', 'MonthlySpend', 'Plan', 'ContractType', 'AutoRenew', 'Region', 'SignupDate', 'Churn']:
    print(col, Data[col].unique()[:15], '\n')

Age ['49.0' '57.0' '41.0' '59.0' '40.0' '30.0' '39.0' '47.0' '28.0' '32.0'
 '26.0' '38.0' '61.0' '31.0' '44.0'] 

MonthlySpend ['4912.97' '2307.58' '984.95' '5447.63' '1348.48' '1127.74' '1660.59'
 '1676.01' '2479.13' '2122.53' '577.39' '4146.31' '1440.01' '6641.5'
 '1488.83'] 

Plan ['Basic' 'Premium' 'Standard' 'Std' 'PREMIUM' 'basic' 'Unknown' nan
 'Enterprise'] 

ContractType ['Yearly' 'Monthly' 'Annual' nan 'Quarterly' 'M' 'monthly'] 

AutoRenew ['No' 'Yes' nan 'yes' 'no'] 

Region ['West' 'Unknown' 'south' 'North' 'East' nan 'South' 'NORTH'] 

SignupDate ['2025-12-15 00:00:00' '2022-06-22 00:00:00' '2020-04-04 00:00:00'
 '2019-05-26 00:00:00' '2021-11-02 00:00:00' '2020-05-11 00:00:00'
 '2022-08-01 00:00:00' '2020-02-02 00:00:00' '2019-11-08 00:00:00'
 '2025-08-05 00:00:00' '2022-02-20 00:00:00' '2023-01-25 00:00:00'
 '2024-09-04 00:00:00' '2021-06-21 00:00:00' '2020-06-18 00:00:00'] 

Churn ['No' 'Yes' 'NO' 'Unknown' 'Active' 'yes' nan 'Churned' 'Y' 'YES' 'N' '0'
 'no' '1'] 



In [7]:
import pandas as pd
import numpy as np

# Numeric columns stored as object (Age, MonthlySpend)
# Strip symbols/commas/text first, then coerce; bad values become NaN
Data['MonthlySpend'] = (Data['MonthlySpend']
                        .astype(str)
                        .str.replace(r'[^0-9.\-]', '', regex=True)
                        .replace('', np.nan))
Data['MonthlySpend'] = pd.to_numeric(Data['MonthlySpend'], errors='coerce')   # float64

Data['Age'] = pd.to_numeric(Data['Age'], errors='coerce')
Data['Age'] = Data['Age'].astype('Int64')   # nullable integer (keeps NaN)

# Categorical columns
for col in ['Plan', 'ContractType', 'Region']:
    Data[col] = Data[col].str.strip().str.title().astype('category')

# Yes/No columns -> boolean (nullable)
yes_no = {'yes': True, 'y': True, '1': True, 'true': True,
          'no': False, 'n': False, '0': False, 'false': False}
for col in ['AutoRenew', 'Churn']:
    Data[col] = (Data[col].astype(str).str.strip().str.lower()
                 .map(yes_no).astype('boolean'))

# Date column
Data['SignupDate'] = pd.to_datetime(Data['SignupDate'], errors='coerce')

# Float columns that are really counts -> nullable integer
count_cols = ['Tenure_Months', 'SupportTickets_90D', 'LoginFrequency_30D',
              'FailedPayments_90D', 'DaysSinceLastContact', 'Complaints_90D']
for col in count_cols:
    Data[col] = Data[col].round().astype('Int64')

# customer_id is an identifier, not a number to do math on
Data['customer_id'] = Data['customer_id'].astype(str)

Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6625 entries, 0 to 6624
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           6625 non-null   object        
 1   Age                   6589 non-null   Int64         
 2   Tenure_Months         6625 non-null   Int64         
 3   Plan                  6220 non-null   category      
 4   MonthlySpend          6597 non-null   float64       
 5   SupportTickets_90D    6625 non-null   Int64         
 6   MonthlyUsage_Hours    6590 non-null   float64       
 7   PaymentHistory_Score  6590 non-null   float64       
 8   LoginFrequency_30D    6605 non-null   Int64         
 9   FailedPayments_90D    6625 non-null   Int64         
 10  ContractType          6232 non-null   category      
 11  AutoRenew             5792 non-null   boolean       
 12  DiscountPercent       6625 non-null   float64       
 13  DaysSinceLastConta

In [8]:
Data[Data['Age'].isnull()]

,customer_id,Age,Tenure_Months,Plan,MonthlySpend,SupportTickets_90D,MonthlyUsage_Hours,PaymentHistory_Score,LoginFrequency_30D,FailedPayments_90D,ContractType,AutoRenew,DiscountPercent,DaysSinceLastContact,Complaints_90D,Region,SignupDate,Churn
77,105485,<NA>,47,NaN,1413.12,4,35.36,80.48,17,1,Monthly,False,15.11,178,1,North,2024-11-12,True
294,103187,<NA>,23,Standard,535.16,0,70.39,77.07,10,1,Quarterly,True,7.98,165,0,East,2025-06-28,False
336,100121,<NA>,20,Standard,1612.06,1,73.04,58.54,11,2,Annual,False,33.28,14,1,North,2021-08-22,False
361,100137,<NA>,32,Basic,3233.19,5,77.86,68.55,12,1,Monthly,False,3.15,57,0,North,2019-05-05,True
492,100668,<NA>,25,Unknown,1077.19,1,42.29,82.65,10,0,Monthly,True,22.70,12,1,NaN,2022-10-12,False
521,105596,<NA>,29,Enterprise,1660.01,3,32.74,100.00,15,3,Monthly,True,3.32,9,2,South,2023-08-10,True
992,103395,<NA>,15,Standard,2547.44,2,50.80,34.01,8,0,NaN,True,8.11,15,0,Unknown,2024-05-04,True
1070,104122,<NA>,14,Basic,6295.15,2,81.86,94.82,5,0,Quarterly,<NA>,13.89,16,0,South,2022-12-24,False
1191,101690,<NA>,38,Enterprise,1210.24,2,24.95,56.08,6,0,Monthly,False,28.67,17,1,NaN,2024-05-11,True
1238,100508,<NA>,14,Premium,1222.85,1,74.39,73.10,2,0,Monthly,<NA>,20.69,88,1,East,2022-05-25,False


In [9]:
Data['Age'].dtype

Int64Dtype()

In [10]:
Age_Mode = Data['Age'].mean()
Data['Age'] = np.where(Data['Age'].isna(),Age_Mode,Data['Age'])

In [11]:
Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6625 entries, 0 to 6624
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           6625 non-null   object        
 1   Age                   6625 non-null   float64       
 2   Tenure_Months         6625 non-null   Int64         
 3   Plan                  6220 non-null   category      
 4   MonthlySpend          6597 non-null   float64       
 5   SupportTickets_90D    6625 non-null   Int64         
 6   MonthlyUsage_Hours    6590 non-null   float64       
 7   PaymentHistory_Score  6590 non-null   float64       
 8   LoginFrequency_30D    6605 non-null   Int64         
 9   FailedPayments_90D    6625 non-null   Int64         
 10  ContractType          6232 non-null   category      
 11  AutoRenew             5792 non-null   boolean       
 12  DiscountPercent       6625 non-null   float64       
 13  DaysSinceLastConta

In [12]:
Data[Data['Plan'].isna()].tail()
Data['Plan'] = np.where(Data['Plan'].isna(),Data['Plan'].mode()[0],Data['Plan'])

In [13]:
Data.isnull().sum()

customer_id               0
Age                       0
Tenure_Months             0
Plan                      0
MonthlySpend             28
SupportTickets_90D        0
MonthlyUsage_Hours       35
PaymentHistory_Score     35
LoginFrequency_30D       20
FailedPayments_90D        0
ContractType            393
AutoRenew               833
DiscountPercent           0
DaysSinceLastContact     36
Complaints_90D            0
Region                  846
SignupDate               50
Churn                    23
dtype: int64

In [14]:
Data = Data.dropna(subset=['MonthlySpend'])

In [15]:
Mean_Cols = ['MonthlyUsage_Hours','PaymentHistory_Score']
for col in Mean_Cols:
    Data[col] = Data[col].fillna(Data[col].mean())

In [16]:
Mode_Cols = ['ContractType','AutoRenew','Region']
for col in Mode_Cols:
    Data[col] = Data[col].fillna(Data[col].mode()[0])

In [17]:
Meadian_Cols = ['LoginFrequency_30D','DaysSinceLastContact']
for col in Meadian_Cols:
    Data[col] = Data[col].fillna(Data[col].median())

In [18]:
Drop_Rows_Cols = ['SignupDate','Churn']
Data= Data.dropna(subset=Drop_Rows_Cols).reset_index(drop=True)

# Model 

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [32]:
Scaler = StandardScaler()

In [54]:
cat_cols = Data[['Plan','ContractType','AutoRenew','Region']]
OHE = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
encoded = OHE.fit_transform(cat_cols)
print(OHE.get_feature_names_out())

encoded_df = pd.DataFrame(encoded, columns=OHE.get_feature_names_out(), index=Data.index)
Data_encoded = pd.concat([Data.drop(columns=cat_cols), encoded_df], axis=1)
Data_encoded.drop(columns='SignupDate',inplace=True)

['Plan_Basic' 'Plan_Enterprise' 'Plan_Premium' 'Plan_Standard' 'Plan_Std'
 'Plan_Unknown' 'ContractType_Annual' 'ContractType_M'
 'ContractType_Monthly' 'ContractType_Quarterly' 'ContractType_Yearly'
 'AutoRenew_0.0' 'AutoRenew_1.0' 'Region_East' 'Region_North'
 'Region_South' 'Region_Unknown' 'Region_West']


In [60]:
X = Data_encoded
y = Data['Churn']
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=42)

In [61]:
Scaled_X_train = Scaler.fit_transform(X_train)
Scaled_X_Test = Scaler.transform(X_test)

In [62]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                            n_jobs=-1, random_state=42),
}

In [63]:
from sklearn.pipeline import Pipeline
fitted = {}
for name, model in models.items():
    model.fit(Scaled_X_train, y_train)    
    fitted[name] = model
    print(f"{name} trained")

Decision Tree trained
Random Forest trained


In [64]:
from sklearn.metrics import classification_report
for name, model in fitted.items():
    print(name)
    print(classification_report(y_test, model.predict(X_test)))

Decision Tree
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       907
         1.0       1.00      1.00      1.00       398

    accuracy                           1.00      1305
   macro avg       1.00      1.00      1.00      1305
weighted avg       1.00      1.00      1.00      1305

Random Forest


C:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
C:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       907
         1.0       1.00      1.00      1.00       398

    accuracy                           1.00      1305
   macro avg       1.00      1.00      1.00      1305
weighted avg       1.00      1.00      1.00      1305

